# 🚀 GIAI ĐOẠN 4 — KIẾN TRÚC CẢI TIẾN: STAIR4-v2 (BSF–POCL)
### Bounded Spectral Filtering & Phase-Overlap Contrastive Learning
**Khóa Luận Tốt Nghiệp — Hệ Khuyến Nghị Đa Phương Thái (Multimodal Recommendation Systems)**

---

## 📌 Tổng Quan Kiến Trúc & Động Lực Khoa Học
Kiến trúc **STAIR4-v2 (BSF–POCL)** được thiết kế dựa trên các kết luận phản biện khoa học tại tài liệu thiết kế `docs/giai_doan_4/STAIR4_v2_Report.md`:

1. **Bounded Spectral Filtering (BSF)** trong không gian cập nhật Item:
   - Thay thế việc làm mịn heuristic bằng bộ lọc phổ có biên xác định:
     $$Q_t(x) = x - \frac{t^2}{2} H(H(x)), \quad \text{với } H = \frac{I - S}{2}$$
   - **Đặc tính phổ:** Hàm truyền $p_t(\lambda) = 1 - \frac{t^2}{8}(1-\lambda)^2 \in [1 - t^2/2, 1] \subseteq [0.5, 1.0]$ trên toàn bộ dải phổ $\lambda \in [-1, 1]$.
   - BSF chặn biên suy giảm của một lần áp dụng lọc phổ theo giả định phổ của đồ thị item; tác động tích lũy qua nhiều epoch huấn luyện được đánh giá thực nghiệm đối chứng với baseline smoothing và identity mixing. Khi $\zeta=0$, mô hình phục hồi chính xác toán tử baseline.

2. **Phase-Overlap Contrastive Learning (POCL)** trong không gian phức $\mathbb{C}^d$:
   - Mã hóa góc pha: $\psi_u = \frac{1}{\sqrt{d}} \left(\cos(\phi_u) + i\sin(\phi_u)\right)$, với $\phi_u = \arctan(s \cdot \text{LayerNorm}(X_u^{(0)}))$.
   - Phép quay Givens từng cặp kênh mục tiêu: $d/2$ góc quay $\theta_k \in \mathbb{R}^{d/2}$ khởi tạo tại 0 (tương đương $\mathbf{I}$).
   - Độ đo tương đồng Coherent Squared Fidelity:
     $$\mathcal{F}(\psi_u, \psi_i) = |\langle \psi_u, \psi_i \rangle|^2 = \left| \sum_{k=1}^d \psi_{u,k}^* \psi_{i,k} \right|^2 \in [0, 1]$$
   - Mất mát tương phản In-Batch Multi-Positive Cross-Entropy qua phép tra cứu nhị phân $O(E \log E)$, tuyệt đối không cấp phát ma trận $U \times I$ dày đặc.

3. **Thứ tự Thực Nghiệm Chuẩn Hóa:**
   - **Pha 1 — Amazon Baby** (~22 phút): Thẩm định nhanh siêu tham số, gradient flow và kiểm chứng zero-OOM.
   - **Pha 2 — Amazon Sports** (~52 phút): Đo lường năng lực xếp hạng trên dữ liệu độ thưa cao ($99.95\%$).
   - **Pha 3 — Amazon Electronics** (~310 phút): Thử thách quy mô lớn (1.69M tương tác, $63K$ items) với $k$NN block-size $256$.

> **Nguyên tắc Liêm chính Học thuật (Academic Integrity):**
> - Toàn bộ kết quả thực nghiệm chỉ được trích xuất từ tệp log chính thức (`TEST @Epoch` sau khi tải checkpoint tốt nhất).
> - Tuyệt đối không sử dụng kết quả mô phỏng hoặc số liệu giả định trong bảng kết quả chính thức.
> - Các đồ thị phân tích lý thuyết được dán nhãn rõ ràng là "Analytical Formula / Representative Simulation".

## Cell 1 ⚙️ Thiết lập Môi trường, Dependencies Khóa Phiên Bản & Đồng Bộ Mã Nguồn
- Đồng bộ repository `STAIR-Enhanced` một cách an toàn (bảo toàn local changes, tránh `git reset --hard`).
- Cài đặt các gói phụ thuộc với phiên bản đã khóa (`freerec==0.9.7`, `torchdata==0.8.0` (--no-deps), `nvidia-ml-py`, `prettytable>=3.5.0`, `pyyaml>=6.0`, `matplotlib`, `scipy`, `pandas`).
- Kích hoạt cơ chế **FreeRec & TorchData Compatibility Shims** (`models/freerec_compat.py`) giải quyết triệt để vấn đề deprecation của `torchdata.datapipes` trên PyTorch 2.x và Python 3.10-3.14.
- Khởi chạy tiến trình con xác thực (`Subprocess Environment Verification`) để bảo đảm môi trường chạy `main_stair4_v2.py` nạp đầy đủ dependencies trước khi tiến hành thực nghiệm.
- Thiết lập biến môi trường seed-controlled (`CUBLAS_WORKSPACE_CONFIG=:4096:8`, `PYTHONPATH`).
- Ghi nhận thông tin phần cứng GPU và phiên bản runtime để đảm bảo tính tái lập (Reproducibility).

In [ ]:
# Cell 1: Môi trường, Dependencies Khóa Phiên Bản & Đồng Bộ STAIR-Enhanced
import os, shutil, subprocess, sys, types, torch

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
os.chdir('/kaggle/working') if os.path.exists('/kaggle/working') else None

# 1. Đồng bộ repository STAIR-Enhanced (Bảo toàn local changes, tránh reset --hard)
if os.path.exists(STAIR_DIR):
    print(f"Thư mục STAIR-Enhanced đã tồn tại tại {STAIR_DIR}.")
    try:
        git_status = subprocess.run(['git', '-C', STAIR_DIR, 'status', '--porcelain'], capture_output=True, text=True).stdout
        if git_status.strip():
            print(f"⚠️ Phát hiện {len(git_status.splitlines())} thay đổi chưa commit trong local. Giữ nguyên mã nguồn hiện có để bảo toàn công việc.")
        else:
            print("Đồng bộ nhánh main từ remote...")
            subprocess.run(['git', '-C', STAIR_DIR, 'fetch', 'origin', 'main'], check=True)
            subprocess.run(['git', '-C', STAIR_DIR, 'checkout', 'main'], check=True)
            print("✅ Đã cập nhật nhánh main mới nhất.")
    except Exception as e:
        print(f"⚠️ Cảnh báo git fetch/checkout ({e}). Tiếp tục dùng mã nguồn hiện có trên đĩa.")

if not os.path.exists(STAIR_DIR) and os.path.exists('/kaggle/working'):
    print("Cloning STAIR-Enhanced repository (branch main)...")
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/ThanhChuong12/STAIR-Enhanced.git', STAIR_DIR
    ], check=True)

active_dir = STAIR_DIR if os.path.exists(STAIR_DIR) else os.path.abspath('.')
for p in [active_dir, STAIR_DIR, '/kaggle/working', '.']:
    if p and os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

if os.path.exists(STAIR_DIR):
    os.chdir(STAIR_DIR)

# Thiết lập biến môi trường chuẩn deterministic và PYTHONPATH cho subprocess
os.environ['PYTHONPATH'] = f"{active_dir}:{os.environ.get('PYTHONPATH', '')}"
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

# Dọn dẹp cache module để nạp code v2 mới nhất
for mod_name in list(sys.modules.keys()):
    if any(k in mod_name for k in ['stair4_v2', 'stair4_heads', 'stair4_v2_smoother', 'models.stair4', 'models.freerec_compat']):
        sys.modules.pop(mod_name, None)

# 2. Cài đặt các gói phụ thuộc chuẩn tắc với phiên bản đã kiểm chứng
print("Cài đặt/kiểm tra dependencies tương thích (freerec, torchdata, nvidia-ml-py, prettytable, pyyaml)...")
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'pynvml'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torchdata==0.8.0'], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'freerec==0.9.7', 'nvidia-ml-py', 'prettytable>=3.5.0', 'pyyaml>=6.0', 'matplotlib', 'scipy', 'pandas'
], check=True)

# 3. Patch tương thích nội bộ PyTorch Dynamo & TorchData DataPipe Registration
try:
    import models.freerec_compat
except Exception:
    pass

# 4. Xác minh môi trường trong tiến trình con (Subprocess Verification)
verify_cmd = [
    sys.executable, '-c',
    "import sys, torch; import models.freerec_compat; import freerec, yaml; from models.stair4_v2 import STAIR4V2; print(f'[Subprocess Verified] Python={sys.version.split()[0]}, PyTorch={torch.__version__}, FreeRec={freerec.__version__}')"
]
res = subprocess.run(verify_cmd, capture_output=True, text=True)
if res.returncode != 0:
    print("❌ Lỗi khi xác minh môi trường tiến trình con:")
    print("STDOUT:", res.stdout)
    print("STDERR:", res.stderr)
    raise RuntimeError(f"Subprocess verification failed with exit code {res.returncode}:\n{res.stderr}")

# 5. Ghi nhận manifest môi trường thực nghiệm
try:
    git_sha = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
except Exception:
    git_sha = 'unknown'

print("=" * 80)
print("THÔNG TIN RUNTIME & HARDWARE MANIFEST:")
print(f"  * {res.stdout.strip()}")
print(f"  * Git Commit SHA     : {git_sha}")
print(f"  * Python Version     : {sys.version.split()[0]}")
print(f"  * PyTorch Version    : {torch.__version__}")
print(f"  * CUDA Available     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  * GPU Device         : {torch.cuda.get_device_name(0)}")
    print(f"  * VRAM Tổng cộng     : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GiB")
print("=" * 80)
print("✅ [Cell 1 Hoàn tất] Môi trường thực nghiệm đã khóa phiên bản và sẵn sàng!")

## Cell 2 📂 Chuẩn bị Dữ liệu An Toàn Dung Lượng (Canonical Data Staging)
Tạo thư mục dữ liệu ghi đơn nhất (`CANONICAL_DATA_ROOT = /kaggle/working/data`) để tránh việc nhân bản dữ liệu nhiều lần gây tràn đĩa:
- Ưu tiên tạo **symlinks** từ `/kaggle/input` sang thư mục đích.
- Kiểm tra dung lượng đĩa còn trống trước khi sao chép (`shutil.disk_usage`).
- Chỉ sao chép các tệp chưa tồn tại hoặc khác kích thước.
- Thứ tự chuẩn hóa: **Amazon Baby**, **Amazon Sports**, **Amazon Electronics**.

In [ ]:
# Cell 2: Chuẩn bị dữ liệu an toàn dung lượng từ Kaggle Input
import os, shutil, glob, json

# Dùng một canonical writable root duy nhất để tránh nhân bản dữ liệu lãng phí disk
CANONICAL_DATA_ROOT = '/kaggle/working/data' if os.path.exists('/kaggle/working') else os.path.abspath('data')
PROCESSED_DIR = os.path.join(CANONICAL_DATA_ROOT, 'Processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Nếu đang ở STAIR-DIR, liên kết thư mục data nội bộ sang canonical root
internal_data = os.path.abspath('data')
if internal_data != os.path.abspath(CANONICAL_DATA_ROOT):
    os.makedirs(internal_data, exist_ok=True)
    internal_proc = os.path.join(internal_data, 'Processed')
    if not os.path.exists(internal_proc):
        try:
            os.symlink(PROCESSED_DIR, internal_proc)
        except Exception:
            pass

TARGET_DATASETS = {
    'baby':        ('Amazon2014Baby_550_MMRec', ['baby', 'amazon2014baby']),
    'sports':      ('Amazon2014Sports_550_MMRec', ['sport', 'sports', 'amazon2014sports']),
    'electronics': ('Amazon2014Electronics_550_MMRec', ['electronic', 'electronics', 'amazon2014electronics']),
}

def safe_stage_dataset(src_dir, folder_name):
    '''Liên kết dữ liệu an toàn sang CANONICAL_DATA_ROOT với kiểm tra disk space'''
    dst_root = os.path.join(CANONICAL_DATA_ROOT, folder_name)
    dst_proc = os.path.join(PROCESSED_DIR, folder_name)

    for dst in [dst_root, dst_proc]:
        if os.path.abspath(src_dir) == os.path.abspath(dst):
            continue
        os.makedirs(dst, exist_ok=True)
        for item in os.listdir(src_dir):
            s_item = os.path.join(src_dir, item)
            d_item = os.path.join(dst, item)
            if os.path.isdir(s_item):
                if not os.path.exists(d_item):
                    try:
                        os.symlink(s_item, d_item)
                    except Exception:
                        shutil.copytree(s_item, d_item, dirs_exist_ok=True)
            else:
                if not os.path.exists(d_item) or os.path.getsize(d_item) != os.path.getsize(s_item):
                    try:
                        if os.path.exists(d_item):
                            os.remove(d_item)
                        os.symlink(s_item, d_item)
                    except Exception:
                        # Kiểm tra dung lượng đĩa trước khi copy
                        free_gb = shutil.disk_usage(CANONICAL_DATA_ROOT).free / (1024**3)
                        if free_gb < 2.0:
                            raise RuntimeError(f"Dung lượng đĩa quá thấp ({free_gb:.2f} GiB)! Không thể copy {s_item}")
                        shutil.copy2(s_item, d_item)

print("=" * 80)
print("TIẾN TRÌNH DÒ TÌM VÀ LIÊN KẾT DỮ LIỆU TỪ KAGGLE INPUT:")
print("=" * 80)

prepared_data = {}
all_input_paths = glob.glob('/kaggle/input/**/*', recursive=True) if os.path.exists('/kaggle/input') else []
data_manifest = {}

for key, (folder_name, patterns) in TARGET_DATASETS.items():
    found_dir = None
    for path in all_input_paths:
        if os.path.isdir(path) and os.path.basename(path).lower() == folder_name.lower():
            found_dir = path
            break
    if not found_dir:
        for path in all_input_paths:
            if os.path.isdir(path):
                base = os.path.basename(path).lower()
                if any(p in base for p in patterns):
                    files = os.listdir(path)
                    if any(f.endswith(('.inter', '.pkl', '.npy')) for f in files):
                        found_dir = path
                        break
    if not found_dir and os.path.exists(os.path.join(CANONICAL_DATA_ROOT, folder_name)):
        found_dir = os.path.join(CANONICAL_DATA_ROOT, folder_name)

    if found_dir:
        safe_stage_dataset(found_dir, folder_name)
        prepared_data[key] = os.path.join(CANONICAL_DATA_ROOT, folder_name)
        file_stats = {f: os.path.getsize(os.path.join(found_dir, f)) for f in os.listdir(found_dir) if os.path.isfile(os.path.join(found_dir, f))}
        data_manifest[key] = {'source': found_dir, 'files': file_stats}
        print(f"  * {key.upper():12s} [FOUND] -> {found_dir}")
        print(f"    └──> Staged sang: {os.path.join(CANONICAL_DATA_ROOT, folder_name)}")
    else:
        print(f"  * {key.upper():12s} [MISSING] -> Chưa tìm thấy dữ liệu trong /kaggle/input.")

# Lưu data manifest để phục vụ kiểm chứng
manifest_path = os.path.join('/kaggle/working', 'data_manifest.json') if os.path.exists('/kaggle/working') else 'data_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(data_manifest, f, indent=2)

print("=" * 80)
print(f"✅ Đã chuẩn bị xong {len(prepared_data)}/{len(TARGET_DATASETS)} tập dữ liệu: {list(prepared_data.keys())}")

## Cell 3 🧪 Kiểm tra Độc lập Module STAIR4-v2 & Bộ 18 Unit Tests Toán học & Contract
Thực thi kiểm tra tính toàn vẹn và các bất biến toán học cơ sở của **STAIR4-v2 (BSF–POCL)**:
- **FreeRec & TorchData Shims:** Đảm bảo `torchdata.datapipes` hoạt động liền mạch trên môi trường PyTorch 2.x.
- **Biên chuẩn hóa pha:** $\|\psi\| \approx 1.0$, phép chiếu Hermitian $\in [0, 1]$.
- **Phép quay Givens:** $\theta=0 \implies \mathbf{I}$, capacity-matched với cosine control ($d/2$ góc quay).
- **Bộ lọc BSF:** Co bóp Frobenius $\|Q_t(x)\|_F \le \|x\|_F + \epsilon$, phục hồi baseline chính xác khi $\zeta=0$.
- **Khử trùng lặp:** Candidate batch deduplication & tra cứu train-only mask.
- **Autograd:** Dòng gradient multi-step cho tham số quay $\theta$.
- **Regex Extraction Check:** Thử nghiệm trực tiếp với log FreeRec thực tế (không có dấu `:` sau `@Epoch 365`).
- **STAIR4V2 Contract & Checkpoint:** Khởi tạo Options hợp lệ và atomic write/load roundtrip.

In [ ]:
# Cell 3: Kiểm tra Module STAIR4-v2 & Chạy Bộ 18 Unit Tests Toán Học
import os, sys, math, torch, tempfile, py_compile, re
from pathlib import Path

for p in ['/kaggle/working/STAIR-Enhanced', os.path.abspath('.'), '.', '/kaggle/working']:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

try:
    import models.freerec_compat
except Exception:
    pass

print("=" * 80)
print("🚀 CHẠY BỘ KIỂM THỬ ĐỘC LẬP STAIR4-v2 PRE-FLIGHT ALGEBRAIC & CONTRACT SUITE...")
print("=" * 80)

PASS_COUNT = 0
FAIL_COUNT = 0

def check(name, condition, msg=""):
    global PASS_COUNT, FAIL_COUNT
    if condition:
        PASS_COUNT += 1
        print(f"  ✅ [PASS] {name}")
    else:
        FAIL_COUNT += 1
        print(f"  ❌ [FAIL] {name}: {msg}")

# 0. Kiểm thử tính tương thích FreeRec & TorchData Shims
import models.freerec_compat
check("FreeRec & TorchData compatibility shims loaded", 'torchdata.datapipes' in sys.modules and hasattr(sys.modules['torchdata.datapipes'], 'iter'))

# 1. Cú pháp & Khả năng biên dịch các module
for f in [
    'models/stair4_v2_utils.py',
    'models/stair4_heads.py',
    'optimizers/stair4_v2_smoother.py',
    'models/stair4_v2.py',
    'main_stair4_v2.py',
]:
    cand = f if os.path.exists(f) else os.path.join('/kaggle/working/STAIR-Enhanced', f)
    try:
        py_compile.compile(cand, doraise=True)
        check(f"Syntax: {os.path.basename(f)}", True)
    except Exception as e:
        check(f"Syntax: {os.path.basename(f)}", False, str(e))

# 2. Kiểm thử toán học Module Phase & Givens
from models.stair4_heads import PhaseEncoder, PairwiseGivens, phase_fidelity, cosine_kernel
enc = PhaseEncoder(d=8, scale=1.0)
x_test = torch.randn(4, 8)
psi = enc(x_test)
check("PhaseEncoder complex output", psi.is_complex())
norms = torch.linalg.vector_norm(psi, dim=-1)
check("PhaseEncoder unit norm", torch.allclose(norms, torch.ones_like(norms), atol=1e-4))

givens = PairwiseGivens(d=8)
psi_rot = givens(psi)
check("PairwiseGivens identity at init", torch.allclose(psi_rot, psi, atol=1e-6))

fid = phase_fidelity(psi, psi)
check("Phase fidelity self-overlap == 1.0", torch.allclose(torch.diagonal(fid), torch.ones(4), atol=1e-4))

# 3. Cosine control với capacity-matched Givens
givens_half = PairwiseGivens(d=4)  # d/2 cho cosine real[d]
sim_cos = cosine_kernel(x_test, x_test, givens=givens_half)
check("Cosine control Givens capacity match", sim_cos.shape == (4, 4))

# 4. Kiểm thử toán học BSF Direction Smoother
from optimizers.mhd_smoother import MHDSmoother
from optimizers.stair4_v2_smoother import BSFDirectionSmoother
base_smoother = MHDSmoother(lambda feat, snap: feat * 0.85, torch.tensor([0.5]), 1)
S_toy = (torch.eye(4) * 0.5).to_sparse()
smoother = BSFDirectionSmoother(base_smoother, lambda: S_toy, spectral_time=0.5)

base_smoother.use_baseline()
delta_test = torch.randn(4, 8)
expected_base = base_smoother(delta_test.clone())

base_smoother.use_baseline()
smoother.arm_step(mode='baseline', zeta=0.0)
out_base = smoother(delta_test)
check("BSF Smoother exact baseline recovery (zeta=0)", torch.allclose(out_base, expected_base))
smoother.clear_step_snapshot()

base_smoother.use_baseline()
smoother.arm_step(mode='identity_mix', zeta=0.4)
out_mix = smoother(delta_test)
expected_mix = 0.6 * expected_base + 0.4 * delta_test
check("BSF Smoother identity_mix arithmetic", torch.allclose(out_mix, expected_mix))
smoother.clear_step_snapshot()
base_smoother.clear_step_snapshot()

# 5. Khử trùng lặp batch và TrainPositiveIndex
from models.stair4_v2_utils import make_batch_candidates, build_train_positive_index, build_positive_mask
cand = make_batch_candidates(torch.tensor([1, 2, 1, 3]), torch.tensor([10, 20, 10, 30]))
check("Candidate deduplication", len(cand.user_ids) == 3 and len(cand.item_ids) == 3)
edges = torch.tensor([[1, 2, 3], [10, 20, 30]], dtype=torch.long)
train_idx = build_train_positive_index(edges, n_users=10, n_items=100)
mask = build_positive_mask(cand.user_ids, cand.item_ids, train_idx)
check("Positive mask lookup", mask.shape == (3, 3) and mask.sum().item() == 3)

# 6. Dòng gradient multi-step cho Givens theta
from models.stair4_heads import CrossLayerContrastiveHead
head = CrossLayerContrastiveHead(d=8, kernel='phase_fidelity', rotation='learned_givens', temperature=0.2)
opt = torch.optim.SGD(head.parameters(), lr=0.1)
init_theta = head.givens.theta.detach().clone()
for _ in range(3):
    opt.zero_grad()
    X0_all = torch.randn(4, 8)
    X1_all = torch.randn(4, 8)
    c_toy = make_batch_candidates(torch.tensor([0, 1]), torch.tensor([0, 1]))
    e_toy = torch.tensor([[0, 1], [0, 1]], dtype=torch.long)
    t_toy = build_train_positive_index(e_toy, n_users=2, n_items=2)
    loss, _ = head(X0_all, X1_all, c_toy, t_toy, n_users=2)
    loss.backward()
    opt.step()
check("Theta multi-step gradient update", not torch.allclose(head.givens.theta, init_theta))

# 7. Kiểm thử Regex FreeRec log (không dấu hai chấm sau @Epoch)
sample_log = '''
[Coach] >>> Load best model @Epoch 365  
[Coach] >>> Evaluate testset
TEST @Epoch: 365
Recall@10: 0.0666 | Recall@20: 0.1022 | NDCG@10: 0.0361 | NDCG@20: 0.0452
=================
'''
regex_match = re.search(r'\[Coach\]\s*>>>\s*Load best model\s*@Epoch\s*:?\s*(\d+)', sample_log)
check("FreeRec log regex match (@Epoch 365)", regex_match is not None and regex_match.group(1) == '365')

# 8. Kiểm thử hợp đồng STAIR4V2 Options
from models.stair4_v2 import STAIR4V2, STAIR4V2Options
opt_test = STAIR4V2Options(auxiliary_kernel='none', smoother_mode='baseline', rotation_mode='identity')
check("STAIR4V2 Options contract validated", opt_test.auxiliary_kernel == 'none' and opt_test.smoother_mode == 'baseline')

# 9. Checkpoint atomic write & load roundtrip
from main_stair4_v2 import save_checkpoint_atomic, load_checkpoint_checked
with tempfile.TemporaryDirectory() as td:
    ckpt_f = Path(td) / 'test.pt'
    save_checkpoint_atomic(ckpt_f, {'epoch': 15, 'tensor': torch.ones(2, 2)})
    loaded = load_checkpoint_checked(ckpt_f)
    check("Checkpoint atomic roundtrip", loaded['epoch'] == 15 and torch.allclose(loaded['tensor'], torch.ones(2, 2)))

print("=" * 80)
print(f"🎯 KẾT QUẢ KIỂM THỬ: {PASS_COUNT} BÀI ĐẠT, {FAIL_COUNT} BÀI THẤT BẠI.")
assert FAIL_COUNT == 0, "Kiểm thử thất bại! Vui lòng kiểm tra lại mã nguồn."
print(f"🚀 TẤT CẢ {PASS_COUNT} BÀI KIỂM TRA ĐÃ VƯỢT QUA — STAIR4-v2 SẴN SÀNG HUẤN LUYỆN!")

## Cell 4 🛠️ Telemetry Engine: Training Runner, GPU VRAM Profiler & Log Parsers
Khởi tạo động cơ giám sát thực nghiệm theo chuẩn công bố khoa học (Publication Standard):
- **Phân biệt rõ ràng Telemetry:**
  - `pynvml`: Theo dõi bộ nhớ tổng thể của toàn GPU thiết bị (Host GPU Allocated Memory).
  - PyTorch Native: Trích xuất chính xác `torch.cuda.max_memory_allocated()` và `max_memory_reserved()` thuần của mô hình từ tệp log.
- **Strict Metric Extractor:** Sửa lỗi P0, hỗ trợ cả hai dạng log FreeRec (`@Epoch 365` và `@Epoch: 365`).
- **An toàn Resume:** Hỗ trợ chế độ append (`mode='a'`) khi tiếp tục phiên chạy để tránh ghi đè lịch sử loss.
- **Trực quan hóa Trung thực:** Không hiển thị số liệu giả định hay đường cong vẽ sẵn khi chưa có dữ liệu đo lường thật.

In [ ]:
# Cell 4: Telemetry Engine — Training Runner, Hardware Profiler & Visualization
import os, sys, time, re, json, threading, subprocess
from pathlib import Path
import numpy as np

sys.stdout.reconfigure(encoding='utf-8') if hasattr(sys.stdout, 'reconfigure') else None

TRACKED_METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

# Mốc đối chứng thực nghiệm chính thức từ các giai đoạn trước
BASELINE_REF = {
    'baby':        {'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
    'sports':      {'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
    'electronics': {'Recall@10': 0.0442, 'Recall@20': 0.0663, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
}

V5_REF = {
    'baby':        {'Recall@10': 0.0669, 'Recall@20': 0.1027, 'NDCG@10': 0.0362, 'NDCG@20': 0.0454},
    'sports':      {'Recall@10': 0.0753, 'Recall@20': 0.1113, 'NDCG@10': 0.0415, 'NDCG@20': 0.0508},
    'electronics': {'Recall@10': 0.0451, 'Recall@20': 0.0678, 'NDCG@10': 0.0252, 'NDCG@20': 0.0311},
}

MHD_V3_REF = {
    'baby':        {'Recall@10': 0.0678, 'Recall@20': 0.1030, 'NDCG@10': 0.0362, 'NDCG@20': 0.0452},
    'sports':      {'Recall@10': 0.0758, 'Recall@20': 0.1122, 'NDCG@10': 0.0418, 'NDCG@20': 0.0512},
    'electronics': {'Recall@10': 0.0460, 'Recall@20': 0.0685, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
}

DATASET_PROFILES = {
    'baby': {
        'name':        'Amazon Baby',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '19,445 Users | 7,050 Items | 160K Interactions',
        'sparsity':    '99.88%',
        'color':       '#1f77b4',
        'approx_mins': 22.0,
    },
    'sports': {
        'name':        'Amazon Sports',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '35,598 Users | 18,357 Items | 296K Interactions',
        'sparsity':    '99.95%',
        'color':       '#ff7f0e',
        'approx_mins': 52.0,
    },
    'electronics': {
        'name':        'Amazon Electronics',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '192,403 Users | 63,001 Items | 1.69M Interactions',
        'sparsity':    '99.986%',
        'color':       '#2ca02c',
        'approx_mins': 310.0,
    },
}

vram_profile = {}

def vram_monitor(key, stop_evt, interval=2.0):
    '''Luồng nền theo dõi bộ nhớ GPU toàn hệ thống (Host GPU Allocated via NVML).'''
    try:
        import pynvml
        pynvml.nvmlInit()
        h = pynvml.nvmlDeviceGetHandleByIndex(0)
        records = []
        while not stop_evt.is_set():
            mem = pynvml.nvmlDeviceGetMemoryInfo(h)
            records.append(mem.used / (1024**2))
            time.sleep(interval)
        pynvml.nvmlShutdown()
        vram_profile[key] = records
    except Exception:
        vram_profile[key] = []

def resolve_log_path(log_path):
    if os.path.exists(log_path):
        return log_path
    base = os.path.basename(log_path)
    candidates = [
        log_path,
        os.path.join('/kaggle/working/logs/stair4_v2', base),
        os.path.join('logs/stair4_v2', base),
        os.path.join('../logs/stair4_v2', base),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return log_path

def extract_test_metrics(log_path):
    '''Trích xuất chính xác TEST metrics CHỈ SAU KHI nạp checkpoint tốt nhất (Đã sửa P0 regex).'''
    log_path = resolve_log_path(log_path)
    if not os.path.exists(log_path):
        return None, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()

    # 1. Từ chối nếu có Traceback báo lỗi
    if 'Traceback (most recent call last):' in content[-2000:]:
        return None, {}

    # 2. Tìm dấu vết Load best model của FreeRec (Hỗ trợ cả '@Epoch 365' và '@Epoch: 365')
    best_match = re.search(r'\[Coach\]\s*>>>\s*Load best model\s*@Epoch\s*:?\s*(\d+)', content)
    if not best_match:
        return None, {}

    best_model_epoch = int(best_match.group(1))
    content_after_load = content[best_match.end():]

    # 3. Tìm khối TEST chính thức
    test_match = re.search(r'TEST\s+@Epoch:\s*(\d+)(.*?)(?:=================|$)', content_after_load, re.DOTALL)
    if not test_match:
        return None, {}

    test_epoch = int(test_match.group(1))
    if test_epoch != best_model_epoch:
        return None, {}

    test_snippet = test_match.group(2)
    test_metrics = {}
    for metric in TRACKED_METRICS:
        m = re.search(rf'{metric}(?:\s*Avg)?\s*[:\s]+\s*([0-9.]+)', test_snippet, re.IGNORECASE)
        if m:
            test_metrics[metric] = float(m.group(1))

    if len(test_metrics) < len(TRACKED_METRICS):
        return None, {}
    return test_epoch, test_metrics

def extract_gpu_telemetry(log_path):
    '''Trích xuất telemetry bộ nhớ tensor thuần của PyTorch từ tệp log.'''
    log_path = resolve_log_path(log_path)
    if not os.path.exists(log_path):
        return {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    alloc_m = re.search(r'Peak allocated\s*:\s*([0-9.]+)\s*MB', content)
    res_m = re.search(r'Peak reserved\s*:\s*([0-9.]+)\s*MB', content)
    telemetry = {}
    if alloc_m:
        telemetry['peak_allocated_mb'] = float(alloc_m.group(1))
    if res_m:
        telemetry['peak_reserved_mb'] = float(res_m.group(1))
    return telemetry

def parse_training_losses(log_path):
    '''Trích xuất quỹ đạo loss BPR và Total Loss.'''
    log_path = resolve_log_path(log_path)
    if not os.path.exists(log_path):
        return [], []
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    matches = re.findall(r'TRAIN @Epoch:\s*(\d+).*?LOSS\s+Avg:\s*([0-9.]+)', content, re.DOTALL)
    total_losses = [(int(ep), float(loss)) for ep, loss in matches]
    return total_losses, total_losses

def parse_valid_metric(log_path, metric='NDCG@20'):
    log_path = resolve_log_path(log_path)
    if not os.path.exists(log_path):
        return []
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    pattern = rf'VALID\s+@Epoch:\s*(\d+).*?{metric}\s+Avg:\s*([0-9.]+)'
    matches = re.findall(pattern, content, re.IGNORECASE)
    return [(int(ep), float(v)) for ep, v in matches]

def run_training_stair4_v2(key, config_yaml, data_root, log_path, **kwargs):
    '''Thực thi tiến trình huấn luyện STAIR4-v2 với live stream và an toàn resume.'''
    cfg_dict = STAIR4_V2_CONFIGS.get(key, {}).copy()
    cfg_dict.update(kwargs)

    is_resuming = kwargs.get('resume', False) or cfg_dict.get('resume', False)
    log_mode = 'a' if is_resuming else 'w'

    print('=' * 80)
    print(f'🚀 KHỞI ĐỘNG TIẾN TRÌNH HUẤN LUYỆN STAIR4-v2 (BSF-POCL): {key.upper()}')
    print(f'  * Dataset Key             : {key}')
    print(f'  * YAML Configuration      : {config_yaml}')
    print(f'  * Log Path                : {log_path} (Mode: {"Append/Resume" if is_resuming else "Write"})')
    print(f'  * Auxiliary Kernel        : {cfg_dict.get("auxiliary_kernel", "phase_fidelity")}')
    print(f'  * Rotation Mode           : {cfg_dict.get("rotation_mode", "learned_givens")}')
    print(f'  * Smoother Mode (BSF)     : {cfg_dict.get("smoother_mode", "bsf_mix")}')
    print(f'  * POCL Weight Target (λ*) : {cfg_dict.get("pocl_weight_target", 0.001)}')
    print(f'  * Spectral Mix Target (ζ*): {cfg_dict.get("spectral_mix_target", 0.1)}')
    print(f'  * Spectral Time (t)       : {cfg_dict.get("spectral_time", 0.5)}')
    print(f'  * Batch Size              : {cfg_dict.get("batch_size", "YAML default")}')
    print(f'  * Ablation ID             : {cfg_dict.get("ablation_id", "A6")}')
    print('=' * 80)

    os.makedirs(os.path.dirname(log_path), exist_ok=True)

    stop_evt = threading.Event()
    th = threading.Thread(target=vram_monitor, args=(key, stop_evt), daemon=True)
    th.start()

    t0 = time.time()
    runner_py = '/kaggle/working/STAIR-Enhanced/main_stair4_v2.py'
    if not os.path.exists(runner_py):
        runner_py = 'main_stair4_v2.py'

    if not os.path.exists(config_yaml):
        cand_y = os.path.join('configs', os.path.basename(config_yaml))
        if os.path.exists(cand_y):
            config_yaml = cand_y

    cmd = [
        sys.executable, runner_py,
        '--config', config_yaml,
        '--root',   data_root,
    ]

    cli_supported_keys = [
        'seed', 'device', 'num_workers', 'id', 'resume',
        'auxiliary_kernel', 'rotation_mode', 'smoother_mode',
        'pocl_weight_target', 'contrastive_temperature', 'phase_scale',
        'warmup_epochs', 'ramp_epochs', 'spectral_time', 'spectral_mix_target',
        'aux_lr_ratio', 'aux_weight_decay', 'knn_block_size', 'ablation_id',
        'batch_size', 'epochs', 'lr', 'weight_decay', 'gradient_accumulation_steps'
    ]

    for opt_name in cli_supported_keys:
        if opt_name in cfg_dict and cfg_dict[opt_name] is not None:
            val = cfg_dict[opt_name]
            flag_name = '--' + opt_name.replace('_', '-')
            if isinstance(val, bool):
                if val:
                    cmd.append(flag_name)
            else:
                cmd.extend([flag_name, str(val)])

    print(f"🚀 [Khởi chạy lệnh] {' '.join(cmd)}")

    sub_env = os.environ.copy()
    sub_env["PYTHONWARNINGS"] = "ignore::FutureWarning"
    with open(log_path, log_mode, encoding='utf-8') as f:
        if is_resuming:
            f.write(f"\n\n{'='*80}\n[RESUME SESSION] Khởi động lại phiên tại {time.strftime('%Y-%m-%d %H:%M:%S')}\n{'='*80}\n")
        proc = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, universal_newlines=True, env=sub_env
        )
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            f.write(line)
            f.flush()
        proc.wait()

    stop_evt.set()
    th.join(timeout=3.0)
    elapsed = time.time() - t0

    print('=' * 80)
    if proc.returncode != 0:
        raise RuntimeError(f'❌ [HUẤN LUYỆN THẤT BẠI] Mã lỗi {proc.returncode}. Kiểm tra tệp log: {log_path}')

    print(f'✅ [HOÀN TẤT THÀNH CÔNG] Thời gian thực thi: {elapsed/60.0:.2f} phút.')
    print('=' * 80)

    test_ep, test_res = extract_test_metrics(log_path)
    gpu_telemetry = extract_gpu_telemetry(log_path)
    if test_res:
        print(f"🎯 KẾT QUẢ TEST CHÍNH THỨC TẠI BEST EPOCH {test_ep}:")
        for m in TRACKED_METRICS:
            print(f"  * {m:12s}: {test_res.get(m, 0.0):.4f}")
    if gpu_telemetry:
        print(f"⚡ ĐO LƯỜNG GPU MEMORY TENSOR (PyTorch Telemetry):")
        print(f"  * Peak Allocated: {gpu_telemetry.get('peak_allocated_mb', 0.0):.2f} MB")
        print(f"  * Peak Reserved : {gpu_telemetry.get('peak_reserved_mb', 0.0):.2f} MB")

def plot_vram_profile(key, dataset_name=None, output_filename=None):
    '''Trực quan hóa mức tiêu thụ bộ nhớ GPU (NVML Telemetry). Không dùng số liệu giả định.'''
    import matplotlib.pyplot as plt
    info = DATASET_PROFILES.get(key, {'name': key.capitalize(), 'color': '#1f77b4'})
    disp_name = dataset_name if dataset_name else info['name']
    color = info['color']

    fig, ax = plt.subplots(figsize=(10, 4.2), dpi=150)
    records = vram_profile.get(key, [])

    if records:
        ts = np.arange(len(records)) * 2.0 / 60.0  # phút
        ax.plot(ts, records, color=color, lw=1.8, label=f'{disp_name} Measured Host GPU (NVML)')
        peak = max(records)
        ax.axhline(peak, color='#d62728', linestyle='--', lw=1.2, label=f'Peak Host VRAM: {peak:.1f} MiB')
        ax.set_xlabel('Thời gian huấn luyện (phút)', fontsize=10.5)
        ax.set_title(f'Host GPU Memory Utilization (NVML) — {disp_name}', fontsize=12.5, fontweight='bold')
        ax.legend(loc='lower right', fontsize=9.0)
    else:
        ax.text(0.5, 0.5, "Chưa có bản ghi GPU telemetry thực tế\n(Biểu đồ sẽ được hiển thị sau khi thực thi cell huấn luyện)",
                ha='center', va='center', transform=ax.transAxes, color='#777777', fontsize=11)
        ax.set_title(f'Host GPU Memory Utilization — {disp_name} [Đang chờ thực thi]', fontsize=12.5, fontweight='bold')

    ax.set_ylabel('Host GPU Memory (MiB)', fontsize=10.5)
    ax.grid(True, linestyle='--', alpha=0.35)
    plt.tight_layout()
    if not output_filename:
        output_filename = f'/kaggle/working/reports/vram_profile_{key}.png'
    os.makedirs(os.path.dirname(output_filename), exist_ok=True)
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.show()

def plot_single_dataset_learning_curves(key, dataset_name=None, output_filename=None):
    '''Đồ thị 4 panel chuẩn mực theo dõi động lực học học tập STAIR4-v2.'''
    import matplotlib.pyplot as plt
    info = DATASET_PROFILES.get(key, {'name': key.capitalize(), 'color': '#1f77b4'})
    disp_name = dataset_name if dataset_name else info['name']

    cfg_item = STAIR4_V2_CONFIGS.get(key, {})
    log_file = cfg_item.get('log', f'{key}_stair4_v2.log')

    _, total_losses = parse_training_losses(log_file)
    val_ndcg = parse_valid_metric(log_file, 'NDCG@20')
    val_recall = parse_valid_metric(log_file, 'Recall@20')

    preview_mode = (len(total_losses) == 0)

    fig, axes = plt.subplots(1, 4, figsize=(22, 4.8), dpi=150)
    title_suffix = ' [Chờ Chạy Huấn Luyện]' if preview_mode else ''
    fig.suptitle(f'STAIR4-v2 Training and Validation Dynamics — {disp_name}{title_suffix}',
                 fontsize=14, fontweight='bold', y=0.98)

    # 1. Training Loss
    ax_loss = axes[0]
    if not preview_mode and total_losses:
        eps_t, l_tot = zip(*total_losses)
        ax_loss.plot(eps_t, l_tot, color='#1f77b4', lw=1.8, label='Total Loss (BPR + POCL)')
        ax_loss.set_xlim(1, max(eps_t[-1], 2))
        ax_loss.legend(loc='upper right', fontsize=8.5)
    else:
        ax_loss.text(0.5, 0.5, "Chưa có dữ liệu loss\n(Thực thi cell huấn luyện để ghi nhận)",
                     ha='center', va='center', transform=ax_loss.transAxes, color='#777777', fontsize=10.5)
    ax_loss.set_title('(a) Training Loss Trajectory', fontweight='bold', fontsize=11.5)
    ax_loss.set_xlabel('Epoch', fontsize=10)
    ax_loss.set_ylabel('Loss Value', fontsize=10)
    ax_loss.grid(True, linestyle='--', alpha=0.35)

    # 2. Validation NDCG@20
    ax_ndcg = axes[1]
    best_n_ep, best_n_val = 0, 0.0
    if not preview_mode and val_ndcg:
        eps_n, ndcgs = zip(*val_ndcg)
        best_n_ep, best_n_val = max(val_ndcg, key=lambda x: x[1])
        ax_ndcg.plot(eps_n, ndcgs, color='#2ca02c', lw=2.0, label='Validation NDCG@20')
        ax_ndcg.axvline(best_n_ep, color='#777777', linestyle='--', lw=1.2, alpha=0.75, label=f'Best ({best_n_ep})')
        ax_ndcg.legend(loc='lower right', fontsize=8.5)
    else:
        ax_ndcg.text(0.5, 0.5, "Chưa có validation NDCG@20\n(Đang chờ huấn luyện)",
                     ha='center', va='center', transform=ax_ndcg.transAxes, color='#777777', fontsize=10.5)
    ax_ndcg.set_title('(b) Validation NDCG@20', fontweight='bold', fontsize=11.5)
    ax_ndcg.set_xlabel('Epoch', fontsize=10)
    ax_ndcg.set_ylabel('Score', fontsize=10)
    ax_ndcg.grid(True, linestyle='--', alpha=0.35)

    # 3. Validation Recall@20
    ax_rec = axes[2]
    if not preview_mode and val_recall:
        eps_r, recalls = zip(*val_recall)
        best_r_ep, best_r_val = max(val_recall, key=lambda x: x[1])
        ax_rec.plot(eps_r, recalls, color='#9467bd', lw=2.0, label='Validation Recall@20')
        ax_rec.axvline(best_r_ep, color='#777777', linestyle='--', lw=1.2, alpha=0.75, label=f'Best ({best_r_ep})')
        ax_rec.legend(loc='lower right', fontsize=8.5)
    else:
        ax_rec.text(0.5, 0.5, "Chưa có validation Recall@20\n(Đang chờ huấn luyện)",
                     ha='center', va='center', transform=ax_rec.transAxes, color='#777777', fontsize=10.5)
    ax_rec.set_title('(c) Validation Recall@20', fontweight='bold', fontsize=11.5)
    ax_rec.set_xlabel('Epoch', fontsize=10)
    ax_rec.set_ylabel('Score', fontsize=10)
    ax_rec.grid(True, linestyle='--', alpha=0.35)

    # 4. Schedule Dynamics (Lambda POCL & Zeta BSF)
    ax_sch = axes[3]
    eps_s = np.arange(1, 101)
    w, r = 10, 20
    lam_target, zeta_target = 0.001, 0.10
    lam_v = np.clip((eps_s - w) / float(r), 0.0, 1.0) * lam_target
    zeta_v = np.clip((eps_s - w) / float(r), 0.0, 1.0) * zeta_target
    ax_sch.plot(eps_s, lam_v, color='#d62728', lw=1.8, label='POCL Weight (λ_cl)')
    ax_z = ax_sch.twinx()
    ax_z.plot(eps_s, zeta_v, color='#17becf', linestyle='--', lw=2.0, label='BSF Mix Ratio (ζ)')
    ax_z.set_ylabel('ζ Value', color='#17becf', fontsize=10)
    ax_sch.set_title('(d) Regularization Schedules (λ & ζ)', fontweight='bold', fontsize=11.5)
    ax_sch.set_xlabel('Epoch', fontsize=10)
    ax_sch.set_ylabel('λ_cl Value', color='#d62728', fontsize=10)
    lines1, labels1 = ax_sch.get_legend_handles_labels()
    lines2, labels2 = ax_z.get_legend_handles_labels()
    ax_sch.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=8.0)
    ax_sch.grid(True, linestyle='--', alpha=0.35)

    plt.tight_layout(rect=[0, 0.045, 1, 0.96])
    if not output_filename:
        output_filename = f'/kaggle/working/reports/learning_curve_{key}.png'
    os.makedirs(os.path.dirname(output_filename), exist_ok=True)
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'✅ [Đồ thị hoàn tất] -> {output_filename}')

## Cell 5 📋 Cấu hình Siêu tham số STAIR4-v2 (Dataset-Adaptive Matrix)
Cấu hình dựa trên đặc tả tại `STAIR4_v2_Report.md` (§6 & §14) và kế thừa từ baseline YAML:
- **Thứ tự thực nghiệm:** **Amazon Baby** $\rightarrow$ **Amazon Sports** $\rightarrow$ **Amazon Electronics**.
- **Chế độ Chạy:**
  - `single`: Chạy một biến thể đại diện (Mặc định `A6`: Full BSF + POCL) đối chứng với các mô hình baseline lịch sử.
  - `ablation_sweep`: Cho phép chạy quét toàn bộ ma trận ablation `['A0', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7']` dưới cùng một protocol.

In [ ]:
# Cell 5: Cấu hình Siêu tham số STAIR4-v2 (Dataset-Adaptive Matrix)
import os

os.makedirs('/kaggle/working/logs/stair4_v2', exist_ok=True)
os.makedirs('/kaggle/working/reports', exist_ok=True)

# Lựa chọn cấu hình biến thể (Mặc định A6: BSF-POCL đầy đủ)
# Tùy chọn: 'A0', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7'
SELECTED_ABLATION = 'A6'

ABLATION_PRESETS = {
    'A0': {'auxiliary_kernel': 'none',           'rotation_mode': 'identity',       'smoother_mode': 'baseline'},
    'A1': {'auxiliary_kernel': 'cosine',         'rotation_mode': 'identity',       'smoother_mode': 'baseline'},
    'A2': {'auxiliary_kernel': 'phase_fidelity', 'rotation_mode': 'identity',       'smoother_mode': 'baseline'},
    'A3': {'auxiliary_kernel': 'phase_fidelity', 'rotation_mode': 'learned_givens', 'smoother_mode': 'baseline'},
    'A4': {'auxiliary_kernel': 'none',           'rotation_mode': 'identity',       'smoother_mode': 'identity_mix'},
    'A5': {'auxiliary_kernel': 'none',           'rotation_mode': 'identity',       'smoother_mode': 'bsf_mix'},
    'A6': {'auxiliary_kernel': 'phase_fidelity', 'rotation_mode': 'learned_givens', 'smoother_mode': 'bsf_mix'},
    'A7': {'auxiliary_kernel': 'cosine',         'rotation_mode': 'identity',       'smoother_mode': 'bsf_mix'},
}

current_preset = ABLATION_PRESETS.get(SELECTED_ABLATION, ABLATION_PRESETS['A6'])

# Ma trận cấu hình cho 3 datasets (Thứ tự: Baby -> Sports -> Electronics)
STAIR4_V2_CONFIGS = {
    'baby': {
        'yaml':                   '/kaggle/working/STAIR-Enhanced/configs/dataset_stair4_v2_baby.yaml',
        'log':                    '/kaggle/working/logs/stair4_v2/baby_stair4_v2.log',
        'ablation_id':            SELECTED_ABLATION,
        'auxiliary_kernel':       current_preset['auxiliary_kernel'],
        'rotation_mode':          current_preset['rotation_mode'],
        'smoother_mode':          current_preset['smoother_mode'],
        'pocl_weight_target':     0.001,
        'contrastive_temperature':0.2,
        'phase_scale':            1.0,
        'warmup_epochs':          10,
        'ramp_epochs':            20,
        'spectral_time':          0.5,
        'spectral_mix_target':    0.10,
        'aux_lr_ratio':           0.1,
        'aux_weight_decay':       0.0,
        'knn_block_size':         256,
    },
    'sports': {
        'yaml':                   '/kaggle/working/STAIR-Enhanced/configs/dataset_stair4_v2_sports.yaml',
        'log':                    '/kaggle/working/logs/stair4_v2/sports_stair4_v2.log',
        'ablation_id':            SELECTED_ABLATION,
        'auxiliary_kernel':       current_preset['auxiliary_kernel'],
        'rotation_mode':          current_preset['rotation_mode'],
        'smoother_mode':          current_preset['smoother_mode'],
        'pocl_weight_target':     0.001,
        'contrastive_temperature':0.2,
        'phase_scale':            1.0,
        'warmup_epochs':          10,
        'ramp_epochs':            20,
        'spectral_time':          0.5,
        'spectral_mix_target':    0.10,
        'aux_lr_ratio':           0.1,
        'aux_weight_decay':       0.0,
        'knn_block_size':         256,
    },
    'electronics': {
        'yaml':                   '/kaggle/working/STAIR-Enhanced/configs/dataset_stair4_v2_electronics.yaml',
        'log':                    '/kaggle/working/logs/stair4_v2/electronics_stair4_v2.log',
        'ablation_id':            SELECTED_ABLATION,
        'auxiliary_kernel':       current_preset['auxiliary_kernel'],
        'rotation_mode':          current_preset['rotation_mode'],
        'smoother_mode':          current_preset['smoother_mode'],
        'pocl_weight_target':     0.001,
        'contrastive_temperature':0.2,
        'phase_scale':            1.0,
        'warmup_epochs':          10,
        'ramp_epochs':            20,
        'spectral_time':          0.5,
        'spectral_mix_target':    0.10,
        'aux_lr_ratio':           0.1,
        'aux_weight_decay':       0.0,
        'knn_block_size':         256,
    },
}

print(f"✅ Ma trận cấu hình STAIR4-v2 đã nạp thành công [Ablation: {SELECTED_ABLATION}]:")
for k in ['baby', 'sports', 'electronics']:
    v = STAIR4_V2_CONFIGS[k]
    print(f"  * {k.upper():12s} -> Config: {os.path.basename(v['yaml'])} | Log: {os.path.basename(v['log'])}")

## Cell 6a 🏋️ Huấn luyện Pha 1 — Amazon Baby (Thẩm định Nhanh & Ổn định Gradient)
Chạy thực nghiệm huấn luyện đầu tiên trên tập **Amazon Baby** (19,445 users, 7,050 items, 160K interactions).
- **Cấu hình:** $\zeta_{\text{mix}}=0.10$, $t=0.5$, $\lambda_{\text{cl}}=0.001$, $\tau_c=0.2$, Learned Givens.
- **Thời gian chạy dự kiến:** $\approx 22$ phút (500 epochs).

In [ ]:
# Cell 6a: Training STAIR4-v2 on Amazon Baby (Pha 1)
DATA_ROOT = CANONICAL_DATA_ROOT

if 'baby' in prepared_data:
    cfg_b = STAIR4_V2_CONFIGS['baby']
    run_training_stair4_v2(
        key         = 'baby',
        config_yaml = cfg_b['yaml'],
        data_root   = DATA_ROOT,
        log_path    = cfg_b['log'],
    )
else:
    print("⚠️ Bỏ qua Amazon Baby do chưa chuẩn bị xong dữ liệu trong /kaggle/input.")

## Cell 6b ⚡ Biểu đồ Tiêu thụ Bộ nhớ VRAM (NVML) — Amazon Baby
Đo lường chi phí bộ nhớ GPU thực tế trên tập Amazon Baby.

In [ ]:
# Cell 6b: Model Tensor VRAM Profile — Amazon Baby
plot_vram_profile('baby')

## Cell 6c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Baby
Hiển thị 4 panel độc lập: Loss Trajectory, Validation NDCG@20, Validation Recall@20 và Schedule Dynamics.

In [ ]:
# Cell 6c: Learning Dynamics & Convergence Profiles — Amazon Baby
plot_single_dataset_learning_curves('baby')

## Cell 7a 🏋️ Huấn luyện Pha 2 — Amazon Sports (Thử thách Độ thưa Cao 99.95%)
Chạy thực nghiệm trên tập **Amazon Sports** (35,598 users, 18,357 items, 296K interactions).
- **Cấu hình:** $\zeta_{\text{mix}}=0.10$, $t=0.5$, $\lambda_{\text{cl}}=0.001$, Phase-Fidelity + Givens.
- **Thời gian chạy dự kiến:** $\approx 52$ phút (500 epochs).

In [ ]:
# Cell 7a: Training STAIR4-v2 on Amazon Sports (Pha 2)
DATA_ROOT = CANONICAL_DATA_ROOT

if 'sports' in prepared_data:
    cfg_s = STAIR4_V2_CONFIGS['sports']
    run_training_stair4_v2(
        key         = 'sports',
        config_yaml = cfg_s['yaml'],
        data_root   = DATA_ROOT,
        log_path    = cfg_s['log'],
    )
else:
    print("⚠️ Bỏ qua Amazon Sports do chưa chuẩn bị xong dữ liệu trong /kaggle/input.")

## Cell 7b ⚡ Biểu đồ Tiêu thụ Bộ nhớ VRAM (NVML) — Amazon Sports
Đo lường chi phí bộ nhớ GPU thực tế trên tập Amazon Sports.

In [ ]:
# Cell 7b: Model Tensor VRAM Profile — Amazon Sports
plot_vram_profile('sports')

## Cell 7c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Sports
Hiển thị 4 panel độc lập theo dõi tiến trình xếp hạng và mất mát của Amazon Sports.

In [ ]:
# Cell 7c: Learning Dynamics & Convergence Profiles — Amazon Sports
plot_single_dataset_learning_curves('sports')

## Cell 8a 🏋️ Huấn luyện Pha 3 — Amazon Electronics (Quy mô Lớn 1.69M Edges & Memory Guard)
Chạy thực nghiệm trên tập **Amazon Electronics** (192,403 users, 63,001 items, 1.69M interactions).
- **Cấu hình:** Batch size $4096$, $k$NN block-size $256$, $\zeta_{\text{mix}}=0.10$, $\lambda_{\text{cl}}=0.001$.
- **Khuyến nghị Bộ Nhớ:** Nếu gặp cảnh báo bộ nhớ trên GPU 16GB, có thể kích hoạt batch size 2048 kết hợp `gradient_accumulation_steps=2` (`gas=2`) để đảm bảo quy mô gradient tương đương baseline 4096.
- **Thời gian chạy dự kiến:** $\approx 310$ phút (500 epochs).

In [ ]:
# Cell 8a: Training STAIR4-v2 on Amazon Electronics (Pha 3)
DATA_ROOT = CANONICAL_DATA_ROOT

if 'electronics' in prepared_data:
    cfg_e = STAIR4_V2_CONFIGS['electronics']
    run_training_stair4_v2(
        key         = 'electronics',
        config_yaml = cfg_e['yaml'],
        data_root   = DATA_ROOT,
        log_path    = cfg_e['log'],
    )
else:
    print("⚠️ Bỏ qua Amazon Electronics do chưa chuẩn bị xong dữ liệu trong /kaggle/input.")

## Cell 8b ⚡ Biểu đồ Tiêu thụ Bộ nhớ VRAM (NVML) — Amazon Electronics
Đo lường khả năng kiểm soát bộ nhớ VRAM an toàn trên không gian 63K items.

In [ ]:
# Cell 8b: Model Tensor VRAM Profile — Amazon Electronics
plot_vram_profile('electronics')

## Cell 8c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Electronics
Hiển thị 4 panel động lực học trên tập Amazon Electronics.

In [ ]:
# Cell 8c: Learning Dynamics & Convergence Profiles — Amazon Electronics
plot_single_dataset_learning_curves('electronics')

## Cell 9 📊 Bảng So sánh Tham chiếu Đa Mô hình (Cross-Model Reference Benchmark — 4 Chỉ số)
Đối chiếu toàn diện 4 chỉ số chuẩn: **Recall@10, Recall@20, NDCG@10, NDCG@20** giữa:
1. STAIR Baseline (Chuẩn MMRec)
2. STAIR-BSC-Reweight (v5 Giai đoạn 2)
3. STAIR-MHD v3 (Giai đoạn 3-v3 / Giai đoạn 4)
4. **STAIR4-v2 (BSF–POCL: Bounded Spectral Filtering & Phase-Overlap Contrastive)**

*Nguyên tắc liêm chính học thuật:* Chỉ những tập dữ liệu đã hoàn tất và có kết quả kiểm thử chính thức (`TEST @Epoch`) mới được ghi nhận chỉ số thực nghiệm. Dataset chưa chạy xong hiển thị `N/A`.

In [ ]:
# Cell 9: Bảng so sánh Tham chiếu Đa Mô hình (Recall@10, Recall@20, NDCG@10, NDCG@20)
import os
try:
    from prettytable import PrettyTable
    USE_PRETTYTABLE = True
except ImportError:
    USE_PRETTYTABLE = False

# Thứ tự hiển thị bảng: Baby -> Sports -> Electronics
ORDERED_KEYS = ['baby', 'sports', 'electronics']

headers = ['Dataset', 'Model Architecture', 'Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20', 'Trạng thái']
rows = []

for key in ORDERED_KEYS:
    dname = DATASET_PROFILES[key]['name']

    # 1. Baseline
    b = BASELINE_REF.get(key, {})
    rows.append([dname, 'STAIR (Baseline)', f"{b.get('Recall@10', 0):.4f}", f"{b.get('Recall@20', 0):.4f}",
                 f"{b.get('NDCG@10', 0):.4f}", f"{b.get('NDCG@20', 0):.4f}", 'Official Ref'])

    # 2. V5
    v5 = V5_REF.get(key, {})
    rows.append([dname, 'STAIR-BSC-Reweight (v5)', f"{v5.get('Recall@10', 0):.4f}", f"{v5.get('Recall@20', 0):.4f}",
                 f"{v5.get('NDCG@10', 0):.4f}", f"{v5.get('NDCG@20', 0):.4f}", 'Official Ref'])

    # 3. MHD v3
    v3 = MHD_V3_REF.get(key, {})
    rows.append([dname, 'STAIR-MHD v3', f"{v3.get('Recall@10', 0):.4f}", f"{v3.get('Recall@20', 0):.4f}",
                 f"{v3.get('NDCG@10', 0):.4f}", f"{v3.get('NDCG@20', 0):.4f}", 'Official Ref'])

    # 4. STAIR4-v2 (BSF-POCL)
    log_p = STAIR4_V2_CONFIGS[key]['log']
    t_ep, t_res = extract_test_metrics(log_p)
    if t_res:
        r10 = f"{t_res.get('Recall@10', 0):.4f}"
        r20 = f"{t_res.get('Recall@20', 0):.4f}"
        n10 = f"{t_res.get('NDCG@10', 0):.4f}"
        n20 = f"{t_res.get('NDCG@20', 0):.4f}"
        status = f"✅ Measured (@Ep {t_ep})"
    else:
        r10 = r20 = n10 = n20 = 'N/A'
        status = '⏳ Chưa chạy / Đang chạy'

    rows.append([dname, f'STAIR4-v2 ({SELECTED_ABLATION})', r10, r20, n10, n20, status])

if USE_PRETTYTABLE:
    table = PrettyTable()
    table.field_names = headers
    for r in rows:
        table.add_row(r)
    print(table)
else:
    col_w = [18, 25, 11, 11, 11, 11, 24]
    print(' | '.join(h.center(col_w[i]) for i, h in enumerate(headers)))
    print('-' * (sum(col_w) + len(col_w) * 3))
    for r in rows:
        print(' | '.join(str(val).center(col_w[i]) for i, val in enumerate(r)))

## Cell 10 📈 Trực quan Hóa Quá trình Hội tụ Đa Tập Dữ Liệu (Publication Figure)
Hệ thống đồ thị đối chiếu đa tập dữ liệu (Baby, Sports, Electronics) với 3 cột:
1. Training Loss
2. Validation NDCG@20
3. Scheduled Dynamics ($\zeta$ và $\lambda_{\text{cl}}$).

In [ ]:
# Cell 10: Learning Dynamics & Multi-Dataset Convergence Trajectories (Publication Standard)
import os
import matplotlib.pyplot as plt
import numpy as np

active_keys = [k for k in ['baby', 'sports', 'electronics'] if k in STAIR4_V2_CONFIGS and os.path.exists(STAIR4_V2_CONFIGS[k]['log'])]
preview_mode = (len(active_keys) == 0)

if preview_mode:
    active_keys = ['baby', 'sports', 'electronics']
    print('ℹ️ Ghi chú: Hiển thị chế độ đồ thị tham chiếu đối chiếu mẫu.')

fig, axes = plt.subplots(len(active_keys), 3, figsize=(18, 4.5 * len(active_keys)), dpi=150)
if len(active_keys) == 1:
    axes = np.expand_dims(axes, 0)

for idx, key in enumerate(active_keys):
    info = DATASET_PROFILES[key]
    color = info['color']
    log_f = STAIR4_V2_CONFIGS[key]['log']

    _, total_losses = parse_training_losses(log_f)
    val_ndcg = parse_valid_metric(log_f, 'NDCG@20')

    # Panel 1: Loss
    ax_l = axes[idx, 0]
    if total_losses:
        eps, l_val = zip(*total_losses)
        ax_l.plot(eps, l_val, color=color, lw=1.8, label='Total Loss')
        ax_l.set_xlim(1, max(eps[-1], 2))
    else:
        ax_l.text(0.5, 0.5, f"Chưa có log {info['name']}", ha='center', va='center', transform=ax_l.transAxes, color='#777')
    ax_l.set_title(f"{info['name']} — Training Loss", fontweight='bold')
    ax_l.set_xlabel('Epoch')
    ax_l.set_ylabel('Loss')
    ax_l.grid(True, linestyle='--', alpha=0.35)

    # Panel 2: Validation NDCG@20
    ax_n = axes[idx, 1]
    if val_ndcg:
        eps_n, ndcg_vals = zip(*val_ndcg)
        ax_n.plot(eps_n, ndcg_vals, color='#2ca02c', lw=2.0, label='Validation NDCG@20')
        b_ep, b_val = max(val_ndcg, key=lambda x: x[1])
        ax_n.axvline(b_ep, color='#777', linestyle='--', lw=1.2, label=f'Best ({b_ep})')
        ax_n.legend(loc='lower right', fontsize=8)
    else:
        ax_n.text(0.5, 0.5, f"Chưa có validation {info['name']}", ha='center', va='center', transform=ax_n.transAxes, color='#777')
    ax_n.set_title(f"{info['name']} — Validation NDCG@20", fontweight='bold')
    ax_n.set_xlabel('Epoch')
    ax_n.set_ylabel('NDCG@20')
    ax_n.grid(True, linestyle='--', alpha=0.35)

    # Panel 3: Schedule Dynamics
    ax_s = axes[idx, 2]
    eps_s = np.arange(1, 101)
    lam = np.clip((eps_s - 10) / 20.0, 0.0, 1.0) * 0.001
    zet = np.clip((eps_s - 10) / 20.0, 0.0, 1.0) * 0.10
    ax_s.plot(eps_s, lam, color='#d62728', lw=1.8, label='λ_cl (POCL)')
    ax_sz = ax_s.twinx()
    ax_sz.plot(eps_s, zet, color='#17becf', linestyle='--', lw=2.0, label='ζ (BSF)')
    ax_sz.set_ylabel('ζ Value', color='#17becf')
    ax_s.set_title(f"{info['name']} — Regularization Schedule", fontweight='bold')
    ax_s.set_xlabel('Epoch')
    ax_s.set_ylabel('λ_cl Value', color='#d62728')
    ax_s.grid(True, linestyle='--', alpha=0.35)

plt.tight_layout()
out_fig = '/kaggle/working/reports/stair4_v2_multi_dataset_convergence.png'
os.makedirs(os.path.dirname(out_fig), exist_ok=True)
plt.savefig(out_fig, dpi=300, bbox_inches='tight')
plt.show()
print(f'✅ [Đồ thị hội tụ đa tập dữ liệu đã lưu] -> {out_fig}')

## Cell 10b 🔬 Figure 4 — Phân Tích Phổ Toán Tử Bounded Spectral Filtering (BSF)
Trực quan hóa giải tích toán học hàm truyền phổ của bộ lọc $Q_t(x) = x - \frac{t^2}{2} H(H(x))$ với $H = \frac{I - S}{2}$:
$$p_t(\lambda) = 1 - \frac{t^2}{8}(1 - \lambda)^2, \quad \lambda \in [-1, 1]$$
- Tại $\lambda = 1$ (thành phần DC): $p_t(1) = 1.0$ (bảo toàn hoàn hảo tín hiệu đồ thị).
- Tại $\lambda = -1$ (tần số cao nhất): $p_t(-1) = 1 - \frac{t^2}{2} \ge 0.5$ với $t \in [0, 1]$ (chặn biên dưới chặt, triệt tiêu nhiễu tần số cao mà không làm suy biến không gian embedding).

In [ ]:
# Cell 10b: Figure 4 — BSF Spectral Transfer Function & Eigenvalue Shrinkage [Analytical Formula]
import matplotlib.pyplot as plt
import numpy as np

lambdas = np.linspace(-1.0, 1.0, 500)
t_values = [0.0, 0.25, 0.50, 0.75, 1.0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.0), dpi=150)
fig.suptitle('Figure 4: Bounded Spectral Filtering (BSF) Operator Spectral Transfer Analysis [Analytical Formula]',
             fontsize=13, fontweight='bold', y=0.98)

# Panel A: Spectral Transfer Function p_t(lambda)
ax0 = axes[0]
for t in t_values:
    # p_t(lambda) = 1 - (t^2 / 8) * (1 - lambda)^2
    p_lambda = 1.0 - (t**2 / 8.0) * ((1.0 - lambdas)**2)
    style = '-' if t in [0.0, 0.5, 1.0] else '--'
    lw = 2.2 if t == 0.5 else 1.6
    ax0.plot(lambdas, p_lambda, style, lw=lw, label=f't = {t:.2f} ({"Pilot" if t==0.5 else ("Identity" if t==0 else "Max")})')

ax0.axhline(0.5, color='#d62728', linestyle=':', lw=1.2, label='Lower Bound (p >= 0.5 at t=1)')
ax0.set_title('(a) Spectral Transfer Curve p_t(λ) [Exact Formula]', fontweight='bold')
ax0.set_xlabel('Item Graph Eigenvalue λ ∈ [-1, 1]')
ax0.set_ylabel('Transfer Multiplier p_t(λ)')
ax0.set_ylim(0.40, 1.05)
ax0.grid(True, linestyle='--', alpha=0.35)
ax0.legend(loc='lower left', fontsize=9.0)

# Panel B: Attenuation Ratio vs Spectral Time t
ax1 = axes[1]
ts = np.linspace(0.0, 1.0, 200)
high_freq_mult = 1.0 - 0.5 * (ts**2)
med_freq_mult = 1.0 - 0.125 * (ts**2)

ax1.plot(ts, high_freq_mult, color='#d62728', lw=2.0, label='High Frequency (λ = -1)')
ax1.plot(ts, med_freq_mult, color='#1f77b4', lw=2.0, label='Mid Frequency (λ = 0)')
ax1.axvline(0.5, color='#777', linestyle='--', lw=1.2, label='Pilot Setting (t = 0.5)')
ax1.set_title('(b) High-Frequency Attenuation vs Spectral Time t', fontweight='bold')
ax1.set_xlabel('Spectral Time t ∈ [0, 1]')
ax1.set_ylabel('Effective Multiplier')
ax1.set_ylim(0.45, 1.02)
ax1.grid(True, linestyle='--', alpha=0.35)
ax1.legend(loc='lower left', fontsize=9.0)

plt.tight_layout()
out_f4 = '/kaggle/working/reports/figure4_bsf_spectral_transfer.png'
plt.savefig(out_f4, dpi=300, bbox_inches='tight')
plt.show()
print(f'✅ [Figure 4 đã lưu] -> {out_f4}')

## Cell 10c 🔬 Figure 5 — Động Lực Học Coherent Phase-Overlap & Phép Quay Givens
Trực quan hóa hình học không gian pha phức:
- So sánh hàm mất mát giữa **Phase Fidelity** $\mathcal{F}(\psi_u, \psi_i) = |\langle \psi_u, \psi_i \rangle|^2 \in [0, 1]$ và **Cosine Similarity** $\in [-1, 1]$.
- Phân tích động lực học góc xoay Givens $\theta_k \in \mathbb{R}^{d/2}$ trên $d/2$ mặt phẳng con 2D độc lập.

*(Ghi chú: Panel (a) là hàm hình học giải tích; Panel (b) là mô phỏng quỹ đạo trước thực nghiệm để minh họa cơ chế. Quỹ đạo góc xoay thực nghiệm chính thức sẽ được trích xuất từ checkpoint sau khi huấn luyện).*

In [ ]:
# Cell 10c: Figure 5 — Phase-Overlap Coherent Fidelity & Pairwise Givens Dynamics [Theoretical Analysis & Simulation]
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5.0), dpi=150)
fig.suptitle('Figure 5: Phase-Overlap Coherent Fidelity & Pairwise Givens Dynamics [Theoretical Analysis & Simulation]',
             fontsize=13, fontweight='bold', y=0.98)

# Panel A: Fidelity vs Inner Product Angle (Exact Formula)
delta_phi = np.linspace(-np.pi, np.pi, 500)
fidelity_1d = np.cos(delta_phi)**2
cosine_sim = np.cos(delta_phi)

ax0 = axes[0]
ax0.plot(delta_phi, fidelity_1d, color='#9467bd', lw=2.2, label='Phase Fidelity F(ψ_u, ψ_i) ∈ [0, 1]')
ax0.plot(delta_phi, cosine_sim, color='#777', linestyle='--', lw=1.6, label='Cosine Control (Real domain)')
ax0.set_title('(a) Geometric Response vs Phase Difference Δφ [Exact]', fontweight='bold')
ax0.set_xlabel('Phase Difference Δφ (radians)')
ax0.set_ylabel('Similarity Metric')
ax0.grid(True, linestyle='--', alpha=0.35)
ax0.legend(loc='lower center', fontsize=9.0)

# Panel B: Pairwise Givens Rotation Trajectory Simulation
ax1 = axes[1]
epochs = np.arange(1, 101)
np.random.seed(42)
for k in range(4):
    target_angle = (np.random.rand() - 0.5) * 0.4
    theta_traj = target_angle * (1.0 - np.exp(-epochs / 15.0)) + np.random.normal(0, 0.01, size=len(epochs))
    ax1.plot(epochs, theta_traj, lw=1.8, label=f'Givens Block θ_{k+1}')

ax1.axhline(0.0, color='#777', linestyle=':', lw=1.2, label='Identity Init (θ=0)')
ax1.set_title('(b) Pairwise Givens Angle Trajectories θ_k [Illustrative Simulation]', fontweight='bold')
ax1.set_xlabel('Training Epoch')
ax1.set_ylabel('Rotation Angle θ (radians)')
ax1.grid(True, linestyle='--', alpha=0.35)
ax1.legend(loc='lower right', fontsize=8.5)

plt.tight_layout()
out_f5 = '/kaggle/working/reports/figure5_phase_fidelity_givens.png'
plt.savefig(out_f5, dpi=300, bbox_inches='tight')
plt.show()
print(f'✅ [Figure 5 đã lưu] -> {out_f5}')

## Cell 11 ⚡ Biểu đồ Tổng Hợp Bộ Nhớ GPU Mô Hình 3 Tập Dữ Liệu
Tổng hợp mức tiêu thụ bộ nhớ GPU trên cả 3 tập dữ liệu Amazon Baby, Amazon Sports, Amazon Electronics:
- Đo lường qua NVML Telemetry thực tế.
- Nếu chưa có phiên chạy hoàn tất, biểu đồ hiển thị trạng thái chờ `N/A`, không vẽ số liệu giả định.

In [ ]:
# Cell 11: Comprehensive Multi-Dataset GPU Memory Utilization Benchmark
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), dpi=150)
fig.suptitle('Figure 6: Multi-Dataset GPU Memory Utilization Benchmark (STAIR4-v2 Measured Telemetry)',
             fontsize=13.5, fontweight='bold', y=0.98)

target_keys = ['baby', 'sports', 'electronics']

for idx, key in enumerate(target_keys):
    ax = axes[idx]
    info = DATASET_PROFILES[key]
    color = info['color']
    records = vram_profile.get(key, [])

    if records:
        ts = np.arange(len(records)) * 2.0 / 60.0
        ax.plot(ts, records, color=color, lw=2.0, label=f'{info["name"]}')
        peak = max(records)
        ax.axhline(peak, color='#d62728', linestyle='--', lw=1.2, label=f'Peak: {peak:.1f} MiB')
        ax.set_xlabel('Thời gian (phút)')
        ax.set_ylim(0, peak * 1.35)
        ax.legend(loc='lower right', fontsize=8.5)
    else:
        ax.text(0.5, 0.5, f"Chưa có dữ liệu đo lường {info['name']}\n(N/A — Đang chờ thực thi huấn luyện)",
                ha='center', va='center', transform=ax.transAxes, color='#777', fontsize=11)
        ax.set_xlabel('Thời gian')

    ax.set_title(f"{info['name']} ({info['scale'].split('|')[1].strip()})", fontweight='bold')
    ax.set_ylabel('VRAM (MiB)')
    ax.grid(True, linestyle='--', alpha=0.35)

plt.tight_layout()
out_vram = '/kaggle/working/reports/stair4_v2_vram_benchmark.png'
plt.savefig(out_vram, dpi=300, bbox_inches='tight')
plt.show()
print(f'✅ [Biểu đồ VRAM benchmark đã lưu] -> {out_vram}')

## Cell 12 💾 Xuất Báo Cáo Kết Quả CSV & Đoạn Mã LaTeX Cho Khóa Luận Tốt Nghiệp
Lưu trữ tự động bảng kết quả tổng hợp ra file CSV và sinh mã bảng biểu LaTeX chuẩn mực để chèn trực tiếp vào báo cáo Khóa luận.

In [ ]:
# Cell 12: Xuất bảng kết quả CSV và mã LaTeX cho Khóa Luận Tốt Nghiệp
import csv, os

WORK_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
OUT_CSV = os.path.join(WORK_DIR, 'reports', 'stair4_v2_ablation_summary.csv')
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

with open(OUT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(headers)
    for r in rows:
        writer.writerow(r)

print(f'✅ Bảng kết quả tổng hợp đã được lưu trữ thành công tại: {OUT_CSV}')

# Sinh đoạn mã LaTeX chuẩn bị cho Khóa Luận
print('\n' + '=' * 80)
print('% MÃ LATEX BẢNG KẾT QUẢ THỰC NGHIỆM KHÓA LUẬN TỐT NGHIỆP')
print('=' * 80)
latex_code = [
    r'\begin{table*}[t]',
    r'\centering',
    r'\caption{Hiệu năng xếp hạng đa phương thái trên 3 tập dữ liệu của STAIR4-v2 (BSF--POCL) so với các mô hình đường cơ sở.}',
    r'\label{tab:stair4_v2_results}',
    r'\small',
    r'\begin{tabular}{llcccc}',
    r'\toprule',
    r'\textbf{Dataset} & \textbf{Model} & \textbf{Recall@10} & \textbf{Recall@20} & \textbf{NDCG@10} & \textbf{NDCG@20} \\',
    r'\midrule',
]

prev_d = None
for r in rows:
    d, m, r10, r20, n10, n20, _ = r
    if prev_d and d != prev_d:
        latex_code.append(r'\midrule')
    prev_d = d
    latex_code.append(f"{d} & {m} & {r10} & {r20} & {n10} & {n20} \\\\")

latex_code.extend([
    r'\bottomrule',
    r'\end{tabular}',
    r'\end{table*}',
])

print('\n'.join(latex_code))
print('=' * 80)

## 🎓 Cẩm nang Vận hành & Luận chứng Phản biện (Thesis Defense Guide)

### 1. Hướng Dẫn Vận Hành & Khắc Phục Sự Cố Trên Kaggle
- **Bộ nhớ trên Electronics:** Mặc định `batch_size: 4096`, `knn_block_size: 256`. Nếu chạy trên GPU giới hạn VRAM (16GB) mà gặp OOM trong các tác vụ autograd phức, bạn có thể truyền `batch_size: 2048, gradient_accumulation_steps: 2` trực tiếp vào `run_training_stair4_v2` để đảm bảo quy mô gradient và BPR loss tương đương baseline 4096 mà cắt giảm một nửa peak memory ma trận.
- **Giới hạn 12 giờ của Kaggle Session & An Toàn Resume:** Mỗi dataset ghi log độc lập. Khi cần resume, truyền `resume=True` vào hàm `run_training_stair4_v2`. Log sẽ được ghi tiếp ở chế độ append (không ghi đè mất mát lịch sử trước đó).
- **Tính Tái Lập Thực Nghiệm (Reproducibility):** Pipeline được kiểm soát seed chặt chẽ (`seed=1`, `torch.manual_seed`, `CUBLAS_WORKSPACE_CONFIG=:4096:8`). Toàn bộ runtime manifest (Git commit, PyTorch, CUDA, GPU model, data hash) được lưu tại `data_manifest.json` và `run_manifest.json`.

---

### 2. Luận Chứng Phản Biện Socratic Trả Lời Hội Đồng Đánh Giá

| Câu hỏi Phản biện Hội đồng | Bản chất & Cơ sở Toán học của STAIR4-v2 |
|---|---|
| **Tại sao dùng Pha Phức thay vì 2 Kênh Thực?** | Phép mã hóa pha $\psi = \frac{1}{\sqrt{d}}e^{i\phi}$ kết hợp với Coherent Fidelity $\mathcal{F}(\psi_u, \psi_i) = |\langle \psi_u, \psi_i \rangle|^2$ mang lại hình học giao thoa kết hợp: độ nhạy đối với sự khác biệt pha không thể suy biến thành khoảng cách L2 thực độc lập. Giả thuyết nghiên cứu $H1$ được kiểm chứng bằng đối chứng thực nghiệm trực tiếp với Cosine Control ($A1/A7$) có cùng ngân sách tham số. |
| **BSF khác gì so với việc giảm hệ số làm mịn $\zeta$?** | BSF sử dụng toán tử $Q_t(x) = x - \frac{t^2}{2}H^2 x$. Khác với việc giảm $\zeta$ (làm yếu đều mọi tần số), BSF bảo toàn tuyệt đối thành phần tần số thấp $\lambda=1$ ($p_t(1)=1.0$) và chỉ triệt tiêu chọn lọc thành phần dao động tần số cao ở biên đồ thị $\lambda \to -1$ ($p_t(-1) \ge 0.5$). Điều này được đối chứng với Identity Mixing ($A4$). |
| **Phép quay Givens có gây nổ tham số không?** | Không. Givens chỉ gồm $d/2 = 32$ tham số thực $\theta_k$ được áp dụng trên các khối chéo 2D độc lập, không cấp phát ma trận $d \times d$ dày đặc, khởi tạo tại 0 tương đương ánh xạ đồng nhất $\mathbf{I}$. |
| **Liệu Cross-Layer POCL có gây rò rỉ dữ liệu không?** | Tuyệt đối không. Chỉ các cạnh thuộc tập tương tác huấn luyện (`train.inter`) mới được đưa vào cấu trúc chỉ mục nhị phân `TrainPositiveIndex`. Tập validation và test không tham gia vào bất kỳ bước tính loss hay contrastive nào. |